# Crude-Seasonality — a quantitative teardown
### Per-month t-stats · Bonferroni · spring-vs-autumn spread · timer race · sub-period split

![Signal: Weak](https://img.shields.io/badge/Signal-Weak-dab617?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Every year?: Busted](https://img.shields.io/badge/Every_year%3F-Busted-8b949e?style=flat-square)

The deep companion to [the notebook for the curious](01_for_the_curious.ipynb). We test crude-oil monthly seasonality on 309 months of tradable data and find a weak, regime-dependent pattern that fails Bonferroni and falls apart as a trading rule.

> Not investment advice. WTI (CL=F) + energy equities (XLE) + 13-week T-bill (^IRX) monthly, 2000-09 → 2026-05, 309 months (Yahoo Finance, daily closes resampled to month-end, grid asserted hole-free). Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (crude_seasonality/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from crude_seasonality import data, strategy as st
from quantlab import repro
d = repro.as_of(data.fetch_data())   # cache-first (examples/verify.py --fetch); hole-free monthly grid
rf = d["tbill"]
timer_cl = st.seasonal_timer(d["crude"], tbill=rf)
bh_cl = st.buy_hold(d["crude"])
timer_xle = st.seasonal_timer(d["energy"], tbill=rf)
bh_xle = st.buy_hold(d["energy"])
ms = st.month_stats(d["crude"])
sa = st.spring_autumn_tstat(d["crude"])


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **Weak** | spring-vs-autumn t = 2.47 on the full sample; fails Bonferroni; insignificant in 2000–2012 |
| Tradability | **Mirage** | XLE timer Sharpe 0.04 vs buy-and-hold 0.35 (excess of T-bill) |
| Every year? | **Busted** | t = 1.42 in 2000–2012; regime-dependent, not unconditional |

> In plain words: a direction with weak evidence, not a reliable calendar law.

## 1 · The claim, steelmanned

- **H₁:** each spring month (Apr–Jun) has a significantly positive mean return.
- **H₂:** the spring group has a significantly higher mean than the autumn group (t-test on pooled spring vs pooled autumn).
- **H₃:** the calendar timer (long spring, short autumn) beats buy-and-hold.
- **H₄:** the pattern is stable across sub-periods.

## 2 · So what? — what rides on each

If H₁–H₄ hold, a simple calendar rule harvests a seasonal premium in crude and energy equities — no fundamental view needed. If they fail, the seasonal narrative is folklore without reliable monetisation.

## 3 · How we'd know — the protocol

One-sample t-stats for each of the 12 calendar months vs 0 (Bonferroni threshold |t| ≈ 3 for α = 0.05/12 ≈ 0.004 at n ≈ 26); Welch two-sample t-test for spring (Apr–Jun) vs autumn (Aug–Nov); timer (long spring, short autumn, T-bill otherwise) vs buy-and-hold, Sharpe in **excess of the T-bill on both legs**; 2000–2012 / 2013-on sub-period split.

## 4 · The teardown

### 4.1 Per-month t-stats (H₁: any spring month significant?)

In [2]:
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
print('CL=F per-month stats (n~26 each):')
print(f'{"Month":6s}  {"Mean":>8s}  {"t-stat":>8s}  {"n":>4s}  Signal?')
bonferroni_t = 3.0  # approx |t| for alpha=0.05/12
for m in range(1,13):
    row = ms.loc[m]
    sig = '|t|>=3 (Bonferroni)' if abs(row['tstat'])>=bonferroni_t else (
          '|t|>=2 (nominal)' if abs(row['tstat'])>=2 else 'noise')
    print(f'{month_names[m-1]:6s}  {row["mean"]*100:+7.1f}%  {row["tstat"]:+8.2f}  {int(row["n"]):4d}  {sig}')

CL=F per-month stats (n~26 each):
Month       Mean    t-stat     n  Signal?
Jan        +2.0%     +1.18    26  noise
Feb        +3.5%     +2.50    26  |t|>=2 (nominal)
Mar        +2.6%     +0.79    26  noise
Apr        +2.3%     +1.25    26  noise
May        +3.0%     +0.74    26  noise
Jun        +3.0%     +2.39    25  |t|>=2 (nominal)
Jul        +0.7%     +0.41    25  noise
Aug        -0.8%     -0.60    25  noise
Sep        -1.2%     -0.71    26  noise
Oct        -1.8%     -0.85    26  noise
Nov        -2.7%     -1.29    26  noise
Dec        +0.3%     +0.13    26  noise


> In plain words: Feb (t = +2.50) and Jun (t = +2.39) nominally pass |t| ≥ 2 but neither passes the Bonferroni-corrected threshold of |t| ≈ 3. **H₁ rejected** — no individual month is robustly significant after adjusting for 12 simultaneous tests.

### 4.2 Spring vs Autumn spread (H₂)

In [3]:
print(f'Spring (Apr-Jun): mean={sa["spring_mean"]*100:.2f}%  n={sa["n_spring"]}')
print(f'Autumn (Aug-Nov): mean={sa["autumn_mean"]*100:.2f}%  n={sa["n_autumn"]}')
print(f'Spread: {(sa["spring_mean"]-sa["autumn_mean"])*100:.2f}%  t={sa["tstat"]:.2f}')
print(f'House bar |t|>=2: {abs(sa["tstat"])>=2}')
print(f'(Bonferroni for 12 groups would require |t|~3; this is a pre-selected 2-group comparison)')

Spring (Apr-Jun): mean=2.76%  n=77
Autumn (Aug-Nov): mean=-1.62%  n=103
Spread: 4.38%  t=2.47
House bar |t|>=2: True
(Bonferroni for 12 groups would require |t|~3; this is a pre-selected 2-group comparison)


> In plain words: the spring-vs-autumn spread clears |t| = 2.47. This earns a `WEAK` signal stamp — the direction is as claimed and statistically detectable — but the multiple-testing context and sub-period decay prevent a `REAL` stamp. **H₂ weakly supported, conditionally.**

### 4.3 Timer vs buy-and-hold on both crude and energy equities (H₃)

In [4]:
rows = {
    'crude timer (CL=F)': st.summary(timer_cl, rf=rf),
    'buy & hold CL=F': st.summary(bh_cl, rf=rf),
    'energy timer (XLE)': st.summary(timer_xle, rf=rf),
    'buy & hold XLE': st.summary(bh_xle, rf=rf),
}
display(pd.DataFrame(rows).T[['cagr','sharpe','vol_ann','max_drawdown','n']].round(3))
print('Sharpe = excess of T-bill (^IRX), both legs, like-for-like')

,cagr,sharpe,vol_ann,max_drawdown,n
crude timer (CL=F),0.123,0.460,0.297,-0.555,309.0
buy & hold CL=F,0.038,0.238,0.385,-0.865,309.0
energy timer (XLE),0.007,0.037,0.189,-0.574,309.0
buy & hold XLE,0.078,0.354,0.251,-0.640,309.0


Sharpe = excess of T-bill (^IRX), both legs, like-for-like


> In plain words: the crude timer beats buy-and-hold CL=F (Sharpe 0.46 vs 0.24) but crude futures have essentially zero real long-run return — the timer is avoiding the worst months of a bad asset. For the investable proxy (XLE), the timer is catastrophically inferior: Sharpe **0.04 vs 0.35**. **H₃ rejected.**

### 4.4 Sub-period stability (H₄)

In [5]:
d.index = pd.DatetimeIndex(d.index)
for lab, yr_range in [('2000-2012', (2000, 2012)), ('2013-on', (2013, 2030))]:
    sl = d[(d.index.year >= yr_range[0]) & (d.index.year <= yr_range[1])]
    r = st.spring_autumn_tstat(sl['crude'])
    print(f'{lab}: spring={r["spring_mean"]*100:.2f}%  autumn={r["autumn_mean"]*100:.2f}%  '
          f't={r["tstat"]:.2f}  n_sp={r["n_spring"]}  n_au={r["n_autumn"]}')
print('\nH4: significant in 2013-on only (t=2.02); absent in 2000-2012 (t=1.42)')

2000-2012: spring=1.76%  autumn=-1.20%  t=1.42  n_sp=36  n_au=51
2013-on: spring=3.64%  autumn=-2.03%  t=2.02  n_sp=41  n_au=52

H4: significant in 2013-on only (t=2.02); absent in 2000-2012 (t=1.42)


> In plain words: the pattern is concentrated in the post-shale era (2013-on), where US shale production created new seasonal supply dynamics. The pre-shale decade (2000–2012) shows insignificant spread (t = 1.42). **H₄ rejected** — regime-dependent, not an unconditional seasonal law.

## 5 · The verdict

H₁ rejected (no Bonferroni-robust month). H₂ weakly supported (t = 2.47 on the full sample). H₃ rejected (XLE timer Sharpe 0.04 vs 0.35). H₄ rejected (pre-shale decade insignificant). → Signal `WEAK`, Tradability `MIRAGE`, "every year"? `BUSTED`.

## 6 · Could you trade it?

No. The XLE timer gives back almost the entire risk-adjusted return of holding energy equities (Sharpe 0.04 vs 0.35). The crude futures timer looks better in isolation but compares a managed strategy against a commodity with near-zero long-run return. The seasonal signal is too soft, too regime-dependent, and too costly in opportunity cost.

## 7 · Going further

Forks: (a) gasoline futures (RB=F) where the seasonal demand is more direct; (b) crack-spread seasonality; (c) condition on EIA crude inventory levels to sharpen the seasonal signal. Companion cross-asset: [49 Black-Gold](../../49-black-gold/).